In [18]:
from ultralytics import YOLO
from PIL import Image
import os
import base64
import requests


In [19]:
def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        # 读取图片文件内容
        image_data = image_file.read()
        # 将图片内容编码为 base64 格式
        base64_encoded = base64.b64encode(image_data)
        # 将 bytes 类型转换为字符串类型
        base64_encoded_str = base64_encoded.decode('utf-8')
        return base64_encoded_str
    
def send_post_request(image_base64):
    # 请求的URL
    url = "http://127.0.0.1:8080/api/tr-run/"

    # 请求参数，包含图片的 base64 值
    data = {
            "img": image_base64,
            "compress":"0"
             }
    try:
        # 发送 HTTP POST 请求
        response = requests.post(url, data=data)
        
        # # 打印响应信息
        # print("Status code:", response.status_code)
        # print("Response text:", response.text)
        return response
        
    except Exception as e:
        print("Error:", e)


def seq_to_filepath(round, id, result_dir):
    if (id == 0):
        filepath = result_dir + str(round) + '.jpg'
    else:
        filepath = result_dir + str(round) + str(id + 1) + '.jpg'
    return filepath

def is_chinese_char(char):
    # 检查字符的 Unicode 编码范围
    return '\u4e00' <= char <= '\u9fff'

def getSingleBookResult(http_json):
    #  分析http返回的数据，返回该数的[标签数字，前七个字符]
    charatcter = ''
    lable_number = -1
    character_count = 0

    for locate, element, conf in http_json['data']['raw_out']:
        if conf > 0.7 and character_count < 7:
            # print(element, conf)
            if is_chinese_char(element):
                charatcter += element
                character_count += 1
        if element.isdigit():
            lable_number = element

    return [lable_number, charatcter]

In [20]:
dir = './img/'
origin_name = '0'
max_width = 400
results_dir = './runs/segment/predict/crops/book/'
model = YOLO('./ex_best.pt')
# round = 1

results = model.predict(source=dir, save_crop=True)


image 1/1 d:\Projects\yolo\back-end\img\7nc67E0uZtiXbf5c1d4358236959b074c1f95b177d23.jpg: 480x640 1 car, 23 books, 1078.6ms
Speed: 7.1ms preprocess, 1078.6ms inference, 28.8ms postprocess per image at shape (1, 3, 480, 640)
Results saved to runs\segment\predict


In [24]:
# Filter results
for round, r in enumerate(results):
    for id, element in enumerate(r.boxes.xywh):
        if(element[2] > max_width):
            if (id == 0):
                filepath = results_dir + str(round) + '.jpg'
            else:
                filepath = results_dir + str(round) + str(id + 1) + '.jpg'
            # os.remove(filepath)
            print(id, element)
            print(filepath)

25 tensor([2039.1620, 1472.4763, 3985.6760, 2041.9398])
./runs/segment/predict/crops/book/026.jpg
22 tensor([2016.0000, 1549.0990, 4032.0000, 2265.1223])
./runs/segment/predict/crops/book/123.jpg


In [25]:
x_coordinate = dict()
for id, element in enumerate(results[1].boxes.xywh):
    x_coordinate[id] = element[0].item()

# 结果排序
d_order=sorted(x_coordinate.items(), key=lambda x:x[1], reverse=False) 
d_order

[(21, 38.04167938232422),
 (5, 145.12771606445312),
 (16, 310.7284240722656),
 (12, 449.70123291015625),
 (15, 630.2791137695312),
 (10, 866.20458984375),
 (9, 1109.03466796875),
 (19, 1283.3603515625),
 (20, 1414.4693603515625),
 (23, 1513.3006591796875),
 (3, 1666.4979248046875),
 (2, 1836.961181640625),
 (22, 2016.0),
 (4, 2020.5703125),
 (0, 2209.680908203125),
 (1, 2421.4501953125),
 (6, 2614.8173828125),
 (8, 2800.91748046875),
 (14, 2955.2294921875),
 (18, 3091.8037109375),
 (17, 3251.779296875),
 (7, 3453.96728515625),
 (13, 3678.51318359375),
 (24, 3843.1015625),
 (11, 3932.791015625)]

In [26]:
all_book_result_in_list_dict = []
for key, value in d_order:
    path = seq_to_filepath(1, key, results_dir)
    if os.path.exists(path):
        all_book_result_in_list_dict.append(
            send_post_request(
                image_to_base64(path)
            ).json())
        print('finish ' + str(key))

finish 21
finish 5
finish 16
finish 12
finish 15
finish 10
finish 9
finish 19
finish 20
finish 23
finish 3
finish 2
finish 4
finish 0
finish 1
finish 6
finish 8
finish 14
finish 18
finish 17
finish 7
finish 13
finish 24
finish 11


In [27]:
last_number = -1
result_with_question = []
result_with_error = []
for id, i in enumerate(all_book_result_in_list_dict):
    this_number_str, character = getSingleBookResult(i)
    this_number = int(this_number_str)
    if this_number == -1:
        result_with_question.append(id)
        continue
    elif last_number == -1:
        last_number = this_number
    elif this_number == last_number or this_number == last_number + 1:
            pass
    else:
         result_with_error.append(id)

    # if label_number == -1:
    last_number = this_number
    print(id, this_number, character)
    


5 0 刀用牛:彩李咏吟著
7 107 人文学术周宁著跨文
9 107 人文学术周宁著跨文
17 110 当代中国文学学
21 5 比较文学与世界
22 5 学堂文田月圣司火
23 112 博博雅大学堂西方文艺理


In [28]:
result_with_question

[0, 1, 2, 3, 4, 6, 8, 10, 11, 12, 13, 14, 15, 16, 18, 19, 20]

In [27]:
result_with_error

[7, 22, 23]

In [29]:
test_result_boxes = results[1].boxes.xywh
print(test_result_boxes[3])

tensor([1666.4979, 1571.9575,  223.3025, 2257.5986])


In [39]:
import cv2

def draw_bounding_box(image_path, errors_coordinates, output_path):

    # 读取图像
    image = cv2.imread(image_path)
    for err in errors_coordinates:
        x_center, y_center, width, height = err.tolist()
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), thickness=20)
    
    cv2.imwrite(output_path, image)

# 示例使用
# 图像文件路径
image_path = "./img/1.jpg"
# 输出文件路径
output_path = "./te.jpg"
# 方框参数



In [40]:
errors_coordinates = []
for err in result_with_error:
    errors_coordinates.append(test_result_boxes[err])
draw_bounding_box(image_path, errors_coordinates, output_path)

In [16]:
import os

# 指定目录路径
directory = './img/'

# 遍历目录下的所有文件
for root, dirs, files in os.walk(directory):
    for id, filename in enumerate(files):
        # 获取文件的绝对路径
        file_path = os.path.join(root, filename)
        print(id, filename.split('.')[-2])
        


0 7nc67E0uZtiXbf5c1d4358236959b074c1f95b177d23


In [22]:
# results = model.predict(source=dir, save_crop=True)
    # Filter results
result_dir = results_dir
for root, dirs, files in os.walk(dir):
    for r, filename in zip(results, files):
        filename_without_suffix = filename.split('.')[-2]
        for id, element in enumerate(r.boxes.xywh):
            if(element[2] > max_width):
                filepath = seq_to_filepath(filename_without_suffix, id, result_dir)
                # os.remove(filepath)
                print(id, element)
                print(filepath)

20 tensor([1939.5178, 1656.6892, 3876.7085, 1793.6450])
./runs/segment/predict/crops/book/7nc67E0uZtiXbf5c1d4358236959b074c1f95b177d2321.jpg


In [24]:
import os 
import shutil
des_path = './runs'
shutil.rmtree(des_path)
# os.removedirs('./runs')

In [25]:
thisdict = dict()
thisdict[2] = 'fdf'

In [29]:
for index, k, v in enumerate(zip(thisdict.items())):
    print(index, k, v)

ValueError: not enough values to unpack (expected 3, got 2)

In [32]:
thisdict.items()

dict_items([(2, 'fdf')])

In [13]:
import base64
import requests
from ultralytics import YOLO
import os
import cv2
import shutil

dir = './img/'
result_dir = './runs/segment/predict/crops/book/'

max_width = 400

model = YOLO('./ex_best.pt')


def save_base64_image(data, file_path):
    try:
        # Extract the base64 encoded image data
        base64_data = data
        # Decode the base64 data
        binary_data = base64.b64decode(base64_data)
        # Write the binary data to a file
        with open(file_path, 'wb') as f:
            f.write(binary_data)
        print("Image saved successfully.")
    except Exception as e:
        print(f"Error: {e}")


def image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        # 读取图片文件内容
        image_data = image_file.read()
        # 将图片内容编码为 base64 格式
        base64_encoded = base64.b64encode(image_data)
        # 将 bytes 类型转换为字符串类型
        base64_encoded_str = base64_encoded.decode('utf-8')
        return base64_encoded_str


def send_post_request(image_base64):
    # 请求的URL
    url = "http://127.0.0.1:8080/api/tr-run/"

    # 请求参数，包含图片的 base64 值
    data = {
        "img": image_base64,
        # "compress": "0"
    }
    try:
        # 发送 HTTP POST 请求
        response = requests.post(url, data=data)
        # # 打印响应信息
        # print("Status code:", response.status_code)
        # print("Response text:", response.text)
        return response
    except Exception as e:
        print("Error:", e)


def seq_to_filepath(filename, id, result_dir):
    if (id == 0):
        filepath = result_dir + str(filename) + '.jpg'
    else:
        filepath = result_dir + str(filename) + str(id + 1) + '.jpg'
    return filepath


def is_chinese_char(char):
    # 检查字符的 Unicode 编码范围
    return '\u4e00' <= char <= '\u9fff'


def getSingleBookResult(http_json):
    #  分析http返回的数据，返回该数的[标签数字，前七个字符]
    charatcter = ''
    num_str = ''
    lable_number = -1
    character_count = 0

    for locate, element, conf in http_json['data']['raw_out']:
        if conf > 0.75 and character_count < 7:
            # print(element, conf)
            if is_chinese_char(element):
                charatcter += element
                character_count += 1
        if conf > 0.8:
            for i in element:
                if i.isdigit():
                    num_str += i
                else:
                    break
            if (len(num_str) > 1):
                try:
                    lable_number = int(num_str)
                except ValueError as e:
                    lable_number = -1

    return [lable_number, charatcter]


def draw_bounding_box(image_path, errors_box, question_box, output_path):

    # 读取图像
    image = cv2.imread(image_path)
    for err in errors_box:
        x_center, y_center, width, height = err
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 0, 255), thickness=20)

    for question in question_box:
        x_center, y_center, width, height = question
        # 计算方框的左上角和右下角坐标
        x1 = int(x_center - width / 2)
        y1 = int(y_center - height / 2)
        x2 = int(x_center + width / 2)
        y2 = int(y_center + height / 2)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), thickness=10)

    cv2.imwrite(output_path, image)


def preProcess():
    # 预处理
    removed_id_dicts = dict()
    sorted_coordinate_dicts = dict()
    dict_coordinate_data = dict()
    results = model.predict(source=dir, save_crop=True)

    # Filter results
    for root, dirs, files in os.walk(dir):
        for r, filename in zip(results, files):
            removed_id = []
            filename_without_suffix = filename.split('.')[-2]
            id_xlabel_dict = dict()

            for id, element in enumerate(r.boxes.xywh):
                if (element[2] > max_width):
                    filepath = seq_to_filepath(
                        filename_without_suffix, id, result_dir)
                    os.remove(filepath)
                    removed_id.append(id)
                    # print(id, element)
                    # print(filepath)
                id_xlabel_dict[id] = element.tolist()

            sorted_coordinate_dicts[filename_without_suffix] = sorted(
                id_xlabel_dict.items(), key=lambda x: x[1][0], reverse=False)
            dict_coordinate_data[filename_without_suffix] = id_xlabel_dict
            removed_id_dicts[filename_without_suffix] = removed_id

    print('check fileter finished...')
    return dict_coordinate_data, sorted_coordinate_dicts, removed_id_dicts

In [6]:
from dao.BaseDao import BaseDao

# 职位数据管理数据库操作类   DAO：database access object
class BookDao(BaseDao):

    # Book salary
    def getBooks(self):
        sql = 'select * from book'
        result = self.execute(sql=sql)
        resultSet = self.fetchall()
        return resultSet
    
def findLabelFromName(resultSet, character):
    for item in resultSet:
        count = 0
        length = len(character)
        for i in character:
            if i in item['label']:
                count += 1
            if count / length > 0.5:
                return item['num_info']
        count = 0
    return -1

In [10]:
print('start whole process...')
results_with_question = dict()
results_with_error = dict()
all_book_result_in_dict = dict()

dict_coordinate_data, sorted_coordinate_dicts, removed_id_dicts = preProcess()

# request and save the result
print('check request ...')

for name, sorted_dict in sorted_coordinate_dicts.items():
        all_book_result_in_list_dict = dict()
        for key, xywh in sorted_dict:
            if key in removed_id_dicts[name]:
                continue
            path = seq_to_filepath(name, key, result_dir)
            if os.path.exists(path):
                all_book_result_in_list_dict[key] = send_post_request(
                    image_to_base64(path)
                ).json()
                # print('finish ' + str(key))
        all_book_result_in_dict[name] = all_book_result_in_list_dict

# check the result
bookDao = BookDao()
resultSet = bookDao.getBooks()
bookDao.close()





start whole process...

image 1/3 d:\Projects\yolo\back-end\img\ertwqtrewq.jpg: 480x640 26 books, 987.7ms
image 2/3 d:\Projects\yolo\back-end\img\erwertret.jpg: 480x640 25 books, 1030.3ms
image 3/3 d:\Projects\yolo\back-end\img\qwewrea.jpg: 480x640 27 books, 959.0ms
Speed: 4.4ms preprocess, 992.3ms inference, 21.6ms postprocess per image at shape (1, 3, 480, 640)
Results saved to runs\segment\predict
check fileter finished...
check request ...


In [14]:
last_number = -1
for name, list_dict in all_book_result_in_dict.items():
    result_with_error = []
    result_with_question = []

    for id, info in list_dict.items():
        this_number_str, character = getSingleBookResult(info)
        try:
            this_number = int(this_number_str)
        except ValueError as e:
            this_number = -1
        
        if this_number == -1 and len(character) > 1:
            this_number = findLabelFromName(resultSet, character)

        print(str(this_number), character)
        if this_number == -1:
            result_with_question.append(id)
            last_number = this_number
            continue
        elif last_number == -1:
            last_number = this_number
            continue
        elif this_number == last_number or this_number == last_number + 1:
            pass
        else:
            result_with_error.append(id)

        # if label_number == -1:
        last_number = this_number


    results_with_question[name] = result_with_question
    results_with_error[name] = result_with_error


print('rewrite to the img')


for name in results_with_question.keys():
    err_boxes = [dict_coordinate_data[name][index] for index in results_with_error[name]]
    ques_boxes = [dict_coordinate_data[name][index] for index in results_with_question[name]]
    img_path = './img/{}.jpg'.format(name)
    output_path = './output/{}.jpg'.format(name)

    draw_bounding_box(img_path, err_boxes, ques_boxes, output_path)

shutil.rmtree('./runs')

247596 君生我已老爱爬
-1 裟
-1 莫笑
-1 
-1 莫笑我
-1 莫笑我胡力下爱
-1 苣笑我胡力下爱
25972 时光银行家钟花
25972 时光银行家钟花
2597259 马时光银行家钟
-1 美
25982598 魔宸彬作品上海
-1 
25982591 魔宸彬作品上海
2598259 美魔宸彬作品上
-1 美驱魔
-1 美美魔宸彬作品
-1 美美宸彬作品
-1 美美宸彬作品
-1 美美呕驱
-1 自美驱魔宸彬作作
-1 
-1 美|美魔宸彬李作铁
-1 美手影
2599 手影
2589 周伟著
2589 黑周著
2590 文江山无恙信应
2591 战神青上杨永峰
2591 战神青上杨永峰
2591 战神卫青上杨永
512591 战神卫青下杨永
522591 战神卫青下杨永
522591 战神卫青下杨永
525902 江山无恙信应亮
-1 情深未完沈高成
-1 情深未成沈高成
-1 情深未成沈高成
2590 长小在江晃山荡无
50 在在晃旯荡的青
-1 在晃荡的青春里
-1 在晃荡的青春里
2594 窃窃狐犭伍思睿著中
475942594 窃窃狐狐伍思任
175942594 窃狐伍思睿著中
-1 窃伍思睿洪著武
475942595 大大洪武篇章朱颜
475952595 大洪武篇章卡朱
-1 洪武篇章朱未颜
-1 
-1 
-1 美善和谐
-1 
-1 后现代书系艺术
-1 现代书系艺术的
59 现代书系艺术的
60 中国当代文艺学
60 中国当代文艺学
60 中国当代文艺学
61 大写作施晓宇著
61 大写作施晓宇著
61 大学写作施晓宇
62 原型理论与文学
62 原型理论与文学
62 原型理论与文学
63 开始写吧影视剧
63 开始写吧影视剧
63 开始写吧影视剧
64 当代学术棱镜译
64 当代学术棱镜译
64 当代学术棱镜译
665 文当代学学术棱
65 文学理论第三版
65 文学理论第三版
65 文学理论第三版
rewrite to the img


In [20]:
from utils.ImageExecute import *
from multiprocessing import Process

def testProcess():
    print('ffffffffff')
    os.mkdir('test')
    


In [22]:
subprocess = Process(target=testProcess, args=())
subprocess.start()
subprocess.join()
print('gggg')

gggg


In [19]:
os.mkdir('gsg')